# Kaggle Web Probe (CPU · Cache & Isolation Diagnostic)

Pure-CPU connectivity, `/kaggle/working/.ct-cache` binary cache, and `/tmp/ct-runtime` isolation probe.

Run via `colabllm kaggle <short_name> up`.

In [ ]:
# @title 0. Setup & Env -- Kaggle dual-zone layout (/kaggle/working/.ct-cache vs 0700 /tmp/ct-runtime)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

print("CT_CELL=0/5:setup", flush=True)
print("CT_STAGE=setup", flush=True)

SERVICE      = os.environ.get("CT_SERVICE", "web-probe")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "18090"))
AUTH         = os.environ.get("CT_AUTH", "caddy")
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "18080"))
AUTH_USER    = os.environ.get("CT_AUTH_USER", "colabllm")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = os.environ.get("CT_WORK_DIR", "/tmp/ct-runtime")
CACHE_DIR = os.environ.get("CT_CACHE_DIR", "/kaggle/working/.ct-cache")
BIN_DIR = os.environ.get("CT_BIN_DIR", f"{CACHE_DIR}/bin")
os.makedirs(WORK, mode=0o700, exist_ok=True)
os.chmod(WORK, 0o700)
os.makedirs(CACHE_DIR, mode=0o755, exist_ok=True)
os.makedirs(BIN_DIR, mode=0o755, exist_ok=True)
if BIN_DIR not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = BIN_DIR + os.pathsep + os.environ.get("PATH", "")

PROCS = {}

def ct_emit_disk_stat(path_str):
    p = path_str
    while not os.path.exists(p) and os.path.dirname(p) != p:
        p = os.path.dirname(p)
    try:
        free_mb = int(shutil.disk_usage(p).free // (1024 * 1024))
    except OSError:
        free_mb = 0
    print(f"CT_DISK_STAT path={path_str} free_mb={free_mb}", flush=True)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to <WORK>/<name>.log (0600)."""
    lp = f"{WORK}/{name}.log"
    fd = os.open(lp, os.O_WRONLY | os.O_CREAT | os.O_APPEND, 0o600)
    os.chmod(lp, 0o600)
    with os.fdopen(fd, "a") as logf:
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

CADDY_VERSION = "2.10.2"

def ct_write_private_text(path, content):
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    with os.fdopen(fd, "w", encoding="utf-8") as handle:
        handle.write(content)
        handle.flush()
        os.fsync(handle.fileno())

def ct_proc_start_time(pid):
    try:
        fields = open(f"/proc/{int(pid)}/stat", "r", encoding="utf-8", errors="replace").read().rsplit(")", 1)[1].split()
        if fields[0] == "Z":
            return None
        return fields[19]
    except (OSError, IndexError, ValueError):
        return None

def ct_stop_marked_proc(name, timeout=10):
    import signal
    marker = f"{WORK}/{name}.proc.json"
    if not os.path.exists(marker):
        return
    try:
        info = json.loads(open(marker, "r", encoding="utf-8").read())
        pid = int(info.get("pid") or 0)
        start_time = info.get("start_time")
    except (OSError, ValueError, TypeError):
        pid, start_time = 0, None
    if pid > 0 and start_time and ct_proc_start_time(pid) == start_time:
        for sig in (signal.SIGTERM, signal.SIGKILL):
            if ct_proc_start_time(pid) != start_time:
                break
            try:
                os.killpg(os.getpgid(pid), sig)
            except (ProcessLookupError, PermissionError, OSError):
                try:
                    os.kill(pid, sig)
                except (ProcessLookupError, OSError):
                    break
            deadline = time.monotonic() + timeout
            while time.monotonic() < deadline:
                if ct_proc_start_time(pid) != start_time:
                    break
                time.sleep(0.1)
    try:
        os.unlink(marker)
    except OSError:
        pass

def ct_spawn_marked(name, args, env=None, cwd=None):
    ct_stop_marked_proc(name)
    proc = ct_spawn(name, args, env=env, cwd=cwd)
    marker = f"{WORK}/{name}.proc.json"
    ct_write_private_text(marker, json.dumps({"pid": proc.pid, "start_time": ct_proc_start_time(proc.pid)}))
    return proc

def _ct_file_sha256(path):
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(65536)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def ct_ensure_caddy():
    import hashlib, platform, tarfile, tempfile
    arch = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")
    dest = f"{BIN_DIR}/caddy"
    status_label = "miss"
    manifest = f"{CACHE_DIR}/manifests/caddy.json" if CACHE_DIR else ""
    if os.path.exists(dest):
        if not manifest:
            return dest
        try:
            meta = json.loads(open(manifest, "r", encoding="utf-8").read())
            if meta.get("artifact") == "caddy" and meta.get("version") == CADDY_VERSION and meta.get("sha256") == _ct_file_sha256(dest):
                print("CT_CACHE_EVENT artifact=caddy status=hit", flush=True)
                return dest
        except (OSError, ValueError):
            pass
        try:
            os.unlink(dest)
        except OSError:
            pass
        status_label = "corrupt_rebuilt"
    asset = f"caddy_{CADDY_VERSION}_linux_{arch}.tar.gz"
    checks_asset = f"caddy_{CADDY_VERSION}_checksums.txt"
    base = f"https://github.com/caddyserver/caddy/releases/download/v{CADDY_VERSION}/"
    os.makedirs(BIN_DIR, exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        tgz = os.path.join(tmp, asset)
        with urllib.request.urlopen(base + asset, timeout=60) as src, open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        checks = urllib.request.urlopen(base + checks_asset, timeout=60).read().decode("utf-8")
        expected = next((line.split()[0] for line in checks.splitlines() if line.split() and line.split()[-1].lstrip("*") == asset), None)
        algo = {64: hashlib.sha256, 128: hashlib.sha512}.get(len(expected or ""))
        if algo is None or algo(open(tgz, "rb").read()).hexdigest() != expected:
            raise RuntimeError(f"Caddy checksum verification failed for {asset}")
        partial = f"{dest}.{os.getpid()}.partial"
        try:
            with tarfile.open(tgz) as tar:
                entry = tar.getmember("caddy")
                with tar.extractfile(entry) as src, open(partial, "wb") as out:
                    shutil.copyfileobj(src, out)
            os.chmod(partial, 0o755)
            os.replace(partial, dest)
        finally:
            if os.path.exists(partial):
                os.unlink(partial)
    if manifest:
        os.makedirs(os.path.dirname(manifest), exist_ok=True)
        tmp_m = f"{manifest}.{os.getpid()}.tmp"
        with open(tmp_m, "w", encoding="utf-8") as mf:
            mf.write(json.dumps({
                "artifact": "caddy",
                "version": CADDY_VERSION,
                "sha256": _ct_file_sha256(dest),
                "bin_path": dest,
                "verified_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }, indent=2) + "\n")
        os.replace(tmp_m, manifest)
        print(f"CT_CACHE_EVENT artifact=caddy status={status_label}", flush=True)
    return dest

def ct_stop_proc(proc, timeout=15):
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=30, grace=60, dead_markers=(), heartbeat=None, heartbeat_interval=120):
    started = time.monotonic()
    last_heartbeat = started
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        now_m = time.monotonic()
        if heartbeat is not None and (heartbeat_interval is None or (now_m - last_heartbeat) >= heartbeat_interval):
            last_heartbeat = now_m
            try:
                heartbeat(int(now_m - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

ct_emit_disk_stat(CACHE_DIR)
ct_emit_disk_stat(WORK)
print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} work={WORK} cache={CACHE_DIR}", flush=True)


In [ ]:
# @title 1. Install & Verify Binaries -- cache caddy in /kaggle/working/.ct-cache/bin
print("CT_CELL=1/5:install", flush=True)
print("CT_STAGE=install", flush=True)
CADDY_BIN = ct_ensure_caddy()
print(f"Verified caddy binary at {CADDY_BIN}", flush=True)


In [ ]:
# @title 2. Start Probe App -- pure-stdlib HTTP connectivity diagnostic on 127.0.0.1:18090
print('CT_CELL=2/5:service', flush=True)
print('CT_STAGE=service', flush=True)
_APP_PATH = f'{WORK}/kaggle_web_probe_app.py'
ct_write_private_text(_APP_PATH, '"""Pure-stdlib loopback HTTP connectivity and cache probe application for Kaggle."""\nfrom __future__ import annotations\n\nfrom functools import partial\nimport html\nimport http.server\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\nimport time\nfrom typing import Any\n\n\n_START_MONO = time.monotonic()\n\n\ndef _load_manifests(cache_dir: str) -> list[dict[str, Any]]:\n    manifest_dir = Path(cache_dir) / "manifests"\n    if not manifest_dir.is_dir():\n        return []\n    items: list[dict[str, Any]] = []\n    for p in sorted(manifest_dir.glob("*.json")):\n        try:\n            data = json.loads(p.read_text(encoding="utf-8"))\n        except (OSError, UnicodeError, ValueError):\n            continue\n        if isinstance(data, dict):\n            items.append(data)\n    return items\n\n\ndef _free_mb(path_str: str) -> int:\n    target = Path(path_str)\n    while not target.exists() and target != target.parent:\n        target = target.parent\n    try:\n        return int(shutil.disk_usage(target).free // (1024 * 1024))\n    except OSError:\n        return 0\n\n\ndef build_health_payload(service_name: str, cache_dir: str, work_dir: str) -> dict[str, Any]:\n    return {\n        "status": "ok",\n        "service": service_name,\n        "provider": "kaggle",\n        "uptime_seconds": int(time.monotonic() - _START_MONO),\n        "cache_dir": cache_dir,\n        "work_dir": work_dir,\n        "free_mb": _free_mb(cache_dir or "/kaggle/working"),\n        "cached_artifacts": [\n            {\n                "artifact": str(item.get("artifact") or ""),\n                "version": str(item.get("version") or ""),\n                "sha256": str(item.get("sha256") or "")[:16],\n                "verified_at": str(item.get("verified_at") or ""),\n            }\n            for item in _load_manifests(cache_dir)\n        ],\n    }\n\n\ndef render_status_html(payload: dict[str, Any]) -> str:\n    svc = html.escape(str(payload["service"]))\n    uptime = html.escape(str(payload["uptime_seconds"]))\n    cache_dir = html.escape(str(payload["cache_dir"]))\n    free_mb = html.escape(str(payload["free_mb"]))\n    rows_html = []\n    for item in payload["cached_artifacts"]:\n        art = html.escape(str(item.get("artifact") or "-"))\n        ver = html.escape(str(item.get("version") or "-"))\n        sha = html.escape(str(item.get("sha256") or "-"))\n        ver_at = html.escape(str(item.get("verified_at") or "-"))\n        rows_html.append(\n            f"<tr><th scope=\\"row\\">{art}</th><td>{ver}</td><td><code>{sha}...</code></td><td>{ver_at}</td></tr>"\n        )\n    if not rows_html:\n        rows_html.append(\'<tr><th scope="row">(none)</th><td>-</td><td>-</td><td>-</td></tr>\')\n    tbody = "\\n          ".join(rows_html)\n    return f"""<!doctype html>\n<html lang="en">\n<head>\n  <meta charset="utf-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1">\n  <meta name="color-scheme" content="light dark">\n  <title>Kaggle Runtime Probe · {svc}</title>\n  <style>\n    :root {{\n      --bg: #f8fafc;\n      --fg: #0f172a;\n      --card: #ffffff;\n      --border: #cbd5e1;\n      --muted: #475569;\n      --accent: #0284c7;\n      color-scheme: light dark;\n    }}\n    @media (prefers-color-scheme: dark) {{\n      :root {{\n        --bg: #0f172a;\n        --fg: #f1f5f9;\n        --card: #1e293b;\n        --border: #334155;\n        --muted: #94a3b8;\n        --accent: #38bdf8;\n      }}\n    }}\n    body {{\n      font-family: system-ui, -apple-system, sans-serif;\n      background: var(--bg);\n      color: var(--fg);\n      margin: 0;\n      padding: 1.5rem;\n      line-height: 1.5;\n    }}\n    main {{ max-inline-size: 56rem; margin-inline: auto; }}\n    .meta {{\n      display: grid;\n      grid-template-columns: repeat(auto-fit, minmax(11rem, 1fr));\n      gap: 0.75rem;\n      margin-block: 1rem 1.5rem;\n    }}\n    .stat {{\n      background: var(--card);\n      border: 1px solid var(--border);\n      border-radius: 0.5rem;\n      padding: 0.75rem 1rem;\n    }}\n    .stat strong {{ display: block; font-size: 0.85rem; color: var(--muted); }}\n    .table-wrapper {{\n      overflow: auto;\n      max-inline-size: 100%;\n      max-block-size: min(420px, 70vh);\n      container-type: inline-size;\n      background: var(--card);\n      border: 1px solid var(--border);\n      border-radius: 0.5rem;\n    }}\n    table {{ width: 100%; border-collapse: separate; border-spacing: 0; text-align: left; }}\n    caption {{ padding: 0.75rem 1rem; font-weight: 600; text-align: left; }}\n    th, td {{ padding: 0.6rem 1rem; border-block-start: 1px solid var(--border); white-space: nowrap; }}\n    thead {{ position: sticky; inset-block-start: 0; z-index: 1; background: var(--card); }}\n    tbody th {{ position: sticky; inset-inline-start: 0; background: var(--card); }}\n    @container (width < 600px) {{\n      table, thead, tbody, tr, th, td {{ display: block; }}\n      thead {{ display: none; }}\n      tr {{ margin-block-end: 0.75rem; border-block-end: 1px solid var(--border); }}\n      tbody th {{ position: sticky; inset-block-start: 0; z-index: 2; font-weight: 600; }}\n      td:nth-child(2)::before {{ content: "Version: "; color: var(--muted); }}\n      td:nth-child(3)::before {{ content: "SHA-256: "; color: var(--muted); }}\n      td:nth-child(4)::before {{ content: "Verified: "; color: var(--muted); }}\n    }}\n  </style>\n</head>\n<body>\n  <main>\n    <h1>Kaggle Web Probe: {svc}</h1>\n    <p>Pure-CPU connectivity, cache, and isolation diagnostic endpoint.</p>\n    <section class="meta" aria-label="Runtime Metrics">\n      <div class="stat"><strong>Status</strong><span>OK</span></div>\n      <div class="stat"><strong>Uptime</strong><span>{uptime}s</span></div>\n      <div class="stat"><strong>Cache Free Space</strong><span>{free_mb} MiB</span></div>\n      <div class="stat"><strong>Cache Dir</strong><code>{cache_dir}</code></div>\n    </section>\n    <div tabindex="0" role="region" aria-labelledby="probe-cache-caption" class="table-wrapper">\n      <table>\n        <caption id="probe-cache-caption">Verified Cache Artifacts</caption>\n        <thead>\n          <tr><th>Artifact</th><th>Version</th><th>SHA-256</th><th>Verified At</th></tr>\n        </thead>\n        <tbody>\n          {tbody}\n        </tbody>\n      </table>\n    </div>\n  </main>\n</body>\n</html>\n"""\n\n\nclass ProbeHandler(http.server.BaseHTTPRequestHandler):\n    protocol_version = "HTTP/1.1"\n\n    def __init__(self, *args: Any, service_name: str, cache_dir: str, work_dir: str, **kwargs: Any) -> None:\n        self.service_name = service_name\n        self.cache_dir = cache_dir\n        self.work_dir = work_dir\n        super().__init__(*args, **kwargs)\n\n    def log_message(self, format: str, *args: Any) -> None:\n        return\n\n    def _send_security_headers(self) -> None:\n        self.send_header("X-Content-Type-Options", "nosniff")\n        self.send_header("X-Frame-Options", "DENY")\n        self.send_header("Referrer-Policy", "no-referrer")\n        self.send_header("Cache-Control", "no-store")\n        self.send_header(\n            "Content-Security-Policy",\n            "default-src \'self\'; style-src \'self\' \'unsafe-inline\'; object-src \'none\'; base-uri \'none\'; frame-ancestors \'none\'",\n        )\n\n    def _send_json(self, status: int, payload: Any) -> None:\n        raw = json.dumps(payload, ensure_ascii=False).encode("utf-8")\n        self.send_response(status)\n        self._send_security_headers()\n        self.send_header("Content-Type", "application/json; charset=utf-8")\n        self.send_header("Content-Length", str(len(raw)))\n        self.send_header("Connection", "close")\n        self.end_headers()\n        if self.command != "HEAD":\n            self.wfile.write(raw)\n        self.close_connection = True\n\n    def _send_html(self, status: int, html_text: str) -> None:\n        raw = html_text.encode("utf-8")\n        self.send_response(status)\n        self._send_security_headers()\n        self.send_header("Content-Type", "text/html; charset=utf-8")\n        self.send_header("Content-Length", str(len(raw)))\n        self.send_header("Connection", "close")\n        self.end_headers()\n        if self.command != "HEAD":\n            self.wfile.write(raw)\n        self.close_connection = True\n\n    def _handle(self) -> None:\n        path = self.path.split("?", 1)[0] or "/"\n        if path == "/v1" or path.startswith("/v1/"):\n            self._send_json(404, {"error": "not_supported"})\n            return\n        if path == "/health" and self.command in ("GET", "HEAD"):\n            payload = build_health_payload(self.service_name, self.cache_dir, self.work_dir)\n            self._send_json(200, payload)\n            return\n        if path == "/" and self.command in ("GET", "HEAD"):\n            payload = build_health_payload(self.service_name, self.cache_dir, self.work_dir)\n            self._send_html(200, render_status_html(payload))\n            return\n        self._send_json(404, {"error": "not_found"})\n\n    def do_GET(self) -> None:\n        self._handle()\n\n    def do_HEAD(self) -> None:\n        self._handle()\n\n    def do_POST(self) -> None:\n        self._handle()\n\n    def do_PUT(self) -> None:\n        self._handle()\n\n    def do_DELETE(self) -> None:\n        self._handle()\n\n\ndef serve_probe(\n    *,\n    host: str = "127.0.0.1",\n    port: int = 18090,\n    service_name: str = "web-probe",\n    cache_dir: str = "/kaggle/working/.ct-cache",\n    work_dir: str = "/tmp/ct-runtime",\n) -> http.server.ThreadingHTTPServer:\n    handler = partial(\n        ProbeHandler,\n        service_name=service_name,\n        cache_dir=cache_dir,\n        work_dir=work_dir,\n    )\n    server = http.server.ThreadingHTTPServer((host, port), handler)\n    server.daemon_threads = True\n    return server\n\n\ndef main() -> int:\n    port = int(os.environ.get("CT_SERVICE_PORT", "18090"))\n    service_name = os.environ.get("CT_SERVICE", "web-probe")\n    cache_dir = os.environ.get("CT_CACHE_DIR", "/kaggle/working/.ct-cache")\n    work_dir = os.environ.get("CT_WORK_DIR", "/tmp/ct-runtime")\n    srv = serve_probe(host="127.0.0.1", port=port, service_name=service_name, cache_dir=cache_dir, work_dir=work_dir)\n    print(f"[web-probe] listening on http://127.0.0.1:{srv.server_address[1]} cache={cache_dir}", flush=True)\n    try:\n        srv.serve_forever(poll_interval=0.5)\n    except KeyboardInterrupt:\n        pass\n    finally:\n        srv.server_close()\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n')
def start_service():
    return ct_spawn_marked('service', [sys.executable, _APP_PATH], env={
        'PYTHONUNBUFFERED': '1',
        'CT_SERVICE': SERVICE,
        'CT_SERVICE_PORT': str(SERVICE_PORT),
        'CT_CACHE_DIR': CACHE_DIR,
        'CT_WORK_DIR': WORK,
    })

PROCS['service'] = (start_service(), start_service)
ct_wait_http(f'http://127.0.0.1:{SERVICE_PORT}/health', timeout=60, ok=lambda c: c == 200)
print(f'web-probe listening on 127.0.0.1:{SERVICE_PORT}', flush=True)


In [ ]:
print('CT_CELL=3/5:proxy', flush=True)
print('CT_STAGE=proxy', flush=True)
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)

def start_caddy():
    caddy_bin = ct_ensure_caddy()
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output([caddy_bin, "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            header_down -Set-Cookie
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    ct_write_private_text(f"{WORK}/Caddyfile", caddyfile)
    return ct_spawn_marked("caddy", [caddy_bin, "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
print('CT_CELL=4/5:tunnel', flush=True)
print('CT_STAGE=tunnel', flush=True)
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

CLOUDFLARED_VERSION = '2026.11.0'
CLOUDFLARED_SHA256 = {
    'amd64': '51f7c68b57ed7269164075d7e789efce5b59355982f06392deeb037d982a51fd',
    'arm64': 'abe54a006acf9faf2d19ee9d953cb0d0804e92f59c5382a162e57b84d086f56f',
}


def _write_private_text(path, content):
    path = Path(path)
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    with os.fdopen(fd, 'w', encoding='utf-8') as handle:
        handle.write(content)
        handle.flush()
        os.fsync(handle.fileno())


def _touch_private_log(path):
    path = Path(path)
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    os.close(fd)


def _file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(65536)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def _check_cached_binary(destination, artifact, version, cache_dir):
    if not destination.exists():
        return None
    if not cache_dir:
        return 'hit'
    manifest = Path(cache_dir) / 'manifests' / f'{artifact}.json'
    if not manifest.is_file():
        return 'miss'
    try:
        meta = json.loads(manifest.read_text(encoding='utf-8'))
        if (
            meta.get('artifact') == artifact
            and meta.get('version') == version
            and meta.get('sha256') == _file_sha256(destination)
        ):
            return 'hit'
    except (OSError, ValueError):
        pass
    destination.unlink(missing_ok=True)
    return 'corrupt_rebuilt'


def _write_manifest(cache_dir, artifact, version, destination):
    if not cache_dir:
        return
    manifest_dir = Path(cache_dir) / 'manifests'
    manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest = manifest_dir / f'{artifact}.json'
    tmp = manifest.with_name(f'.{artifact}.{os.getpid()}.tmp')
    payload = {
        'artifact': artifact,
        'version': version,
        'sha256': _file_sha256(destination),
        'bin_path': str(destination),
        'verified_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    tmp.write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')
    os.replace(tmp, manifest)


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version, *, cache_dir=None):
    destination = Path(destination)
    cache_state = _check_cached_binary(destination, 'frpc', version, cache_dir)
    if cache_state == 'hit':
        if cache_dir:
            print(f'CT_CACHE_EVENT artifact=frpc status=hit', flush=True)
        return str(destination)
    status_label = cache_state or 'miss'
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            partial=destination.with_name(f'{destination.name}.{os.getpid()}.partial')
            try:
                with tar.extractfile(entry) as source, open(partial, 'wb') as out_bin:
                    shutil.copyfileobj(source, out_bin)
                partial.chmod(0o755)
                os.replace(partial, destination)
            finally:
                partial.unlink(missing_ok=True)
    _write_manifest(cache_dir, 'frpc', version, destination)
    if cache_dir:
        print(f'CT_CACHE_EVENT artifact=frpc status={status_label}', flush=True)
    return str(destination)


def install_cloudflared(destination, *, cache_dir=None, fetch_fn=None):
    destination = Path(destination)
    cf_arch = {'x86_64': 'amd64', 'aarch64': 'arm64'}.get(platform.machine(), 'amd64')
    url = f'https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-{cf_arch}'
    expected_sha = CLOUDFLARED_SHA256.get(cf_arch)
    if not expected_sha:
        raise ValueError(f'Unsupported cloudflared architecture: {platform.machine()}')
    if cache_dir:
        cache_state = _check_cached_binary(destination, 'cloudflared', CLOUDFLARED_VERSION, cache_dir)
        if cache_state == 'hit':
            print('CT_CACHE_EVENT artifact=cloudflared status=hit', flush=True)
            return str(destination)
        status_label = cache_state or 'miss'
        if destination.exists() and _file_sha256(destination) != expected_sha:
            destination.unlink(missing_ok=True)
            status_label = 'corrupt_rebuilt'
        destination.parent.mkdir(parents=True, exist_ok=True)
        partial = destination.with_name(f'{destination.name}.{os.getpid()}.partial')
        try:
            with urllib.request.urlopen(url, timeout=60) as src, open(partial, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            if _file_sha256(partial) != expected_sha:
                raise ValueError('cloudflared checksum verification failed')
            partial.chmod(0o755)
            os.replace(partial, destination)
        finally:
            partial.unlink(missing_ok=True)
        _write_manifest(cache_dir, 'cloudflared', CLOUDFLARED_VERSION, destination)
        print(f'CT_CACHE_EVENT artifact=cloudflared status={status_label}', flush=True)
        return str(destination)
    if fetch_fn is not None:
        return fetch_fn(str(destination), url)
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        partial = destination.with_name(f'{destination.name}.{os.getpid()}.partial')
        try:
            with urllib.request.urlopen(url, timeout=60) as src, open(partial, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            if _file_sha256(partial) != expected_sha:
                raise ValueError('cloudflared checksum verification failed')
            partial.chmod(0o755)
            os.replace(partial, destination)
        finally:
            partial.unlink(missing_ok=True)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx.get('WORK') or os.environ.get('CT_WORK_DIR') or '/content/ct')
    work.mkdir(parents=True, exist_ok=True)
    bin_dir_explicit = ctx.get('BIN_DIR') or os.environ.get('CT_BIN_DIR') or ''
    bin_dir = Path(bin_dir_explicit or '/usr/local/bin')
    cache_dir = ctx.get('CACHE_DIR') or os.environ.get('CT_CACHE_DIR') or ''
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if ctx.get('ct_stop_marked_proc') is not None:
        stop_fn = ctx['ct_stop_marked_proc']
        stop_fn('tunnel-process')
    elif marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        _touch_private_log(work/(name+'.log'))
        proc=ctx['ct_spawn'](name,args)
        _write_private_text(marker, json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        _write_private_text(ca_path, os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        _write_private_text(config_path, render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        frpc_name = f"frpc-{os.environ['CT_FRP_VERSION']}"
        frpc_dest = (bin_dir / frpc_name) if bin_dir_explicit else ((work / frpc_name) if (work / frpc_name).exists() else (bin_dir / frpc_name))
        binary=install_frpc(frpc_dest, os.environ['CT_FRP_VERSION'], cache_dir=cache_dir or None)
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        binary=install_cloudflared(bin_dir / 'cloudflared', cache_dir=cache_dir or None, fetch_fn=ctx.get('ct_fetch_binary'))
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see {work}/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Verify Auth Gate, Signal CT_READY & Keep-Alive Watchdog
import base64
print("CT_CELL=5/5:watchdog", flush=True)
print("CT_STAGE=running", flush=True)

def _kp_log(msg):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] [web-probe] {msg}", flush=True)

def _kp_public_probe():
    creds = base64.b64encode(f"{AUTH_USER}:{AUTH_PASS}".encode("utf-8")).decode("ascii")
    req = urllib.request.Request(
        PUBLIC_URL.rstrip("/") + "/health",
        headers={"Authorization": f"Basic {creds}", "User-Agent": "Mozilla/5.0"},
    )
    with urllib.request.urlopen(req, timeout=20) as resp:
        data = json.loads(resp.read().decode("utf-8"))
        if data.get("status") != "ok":
            raise RuntimeError(f"Public /health returned unexpected payload: {data}")
        return resp.status

def _kp_verify_auth():
    req_unauth = urllib.request.Request(PUBLIC_URL.rstrip("/") + "/", headers={"User-Agent": "Mozilla/5.0"})
    try:
        urllib.request.urlopen(req_unauth, timeout=15)
        raise RuntimeError("Unauthenticated GET / unexpectedly succeeded!")
    except urllib.error.HTTPError as err:
        if err.code != 401:
            raise RuntimeError(f"Expected HTTP 401 without credentials, got {err.code}")
        if err.headers.get("Set-Cookie"):
            raise RuntimeError("Public 401 leaked Set-Cookie")

ct_public_gate(PUBLIC_URL, _kp_public_probe, _kp_verify_auth, _kp_log, timeout=180, interval=8)
if not CREDS_FROM_ENV:
    print(f"CT_CREDENTIALS user={AUTH_USER} pass={AUTH_PASS}")
    print(f"API token: {API_TOKEN}")
print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

def _kp_local_health():
    with urllib.request.urlopen(f"http://127.0.0.1:{SERVICE_PORT}/health", timeout=10) as resp:
        return resp.status == 200 and json.loads(resp.read().decode("utf-8")).get("status") == "ok"

ct_watchdog(
    _kp_local_health,
    start_service,
    _kp_log,
    service_key="service",
    log_name="service",
    interval=30,
    grace=60,
    heartbeat_interval=120,
    heartbeat=lambda up_s: _kp_log(f"heartbeat | up={up_s}s | free_mb={int(shutil.disk_usage(CACHE_DIR).free // (1024 * 1024))}"),
)
